# Sartorius iQue Screener PLUS

The iQue Screener PLUS is a plate-based flow cytometer. Its probe samples the wells of a 96-, 384- or 1536-well plate one after another, with a bubble of air between two wells, into one continuous stream of events; the wells are found in the stream afterwards by when they were sampled. PyLabRobot reads **plates** and the **tube holder**, assigns the events to **wells**, writes **FCS 3.0** files of the plate and of each well, and runs the instrument's **start-up**, **shut-down** and **cleaning** routines and its **deck-height calibration**.

```{device-card} sartorius-ique-screener-plus
```

```{warning}
This driver has **not been run on an iQue Screener PLUS** yet: it is tested against a simulator of the instrument's controller. `setup()` logs a warning. Supervise the first runs, and if you verify it on your instrument, please [report the result](../../needs-testing.md) or open a pull request to update this warning.
```

## Properties

| | |
|---|---|
| Samples | 96-, 384- and 1536-well plates (1536 on instruments that support them); one 1.5 mL microcentrifuge tube in the tube holder |
| Optical configurations | VBR: violet, blue and red lasers, 13 fluorescence channels; VYB: violet, yellow and blue lasers, 13 channels; BR: blue and red lasers, 6 channels |
| Event parameters | height and area of forward scatter, side scatter and every fluorescence channel, pulse width, time |
| Plate handling | plate rail carrying the plate holder; plate shaker up to 3000 rpm (5000 rpm on the HD model) |
| Fluidics | peristaltic sample pump; Buffer, Decon, Clean and Water stations on the deck; Sheath, PLUS One, PLUS Two and Waste bottles on the bottle station |
| Data | the controller's FCS 3.0 file of the plate, and one FCS file per well |
| Interface | Ethernet to the instrument's internal computer, TCP ports 9002 and 9003 |
| Size | 130 x 63 x 66 cm (width x depth x height) with the bottle station, without the computer and the UPS |
| Not supported | firmware updates, teaching plate corners, stations and plate models, well identification with marker beads |

## How it talks

The instrument's internal computer runs a controller service, which drives the sampler, the pump, the shaker, the plate rail and the detector. PyLabRobot is a network client of that service, so no vendor software runs on the PyLabRobot computer. It connects to two endpoints of the controller over TCP, both speaking WCF's binary net.tcp protocol: `/Controller` on port 9002 for requests and for the status updates and event data the controller sends back, and `/ControllerStreaming` on port 9003 for files (the calibration file, and the FCS file of a run). A watchdog service, which starts and stops the controller service, listens on ports 9000 and 9001.

The controller runs a read or a maintenance routine as a worklist: a list of commands that it executes on its own and hands back with the start and end time of each. PyLabRobot builds the worklist from a method's arguments and the instrument's calibration, follows its progress, collects the events as they stream in, and downloads the controller's FCS file at the end.

The controller serves one client at a time. Close any other client software, for example the instrument's own software, before you connect: `setup()` refuses while another client is connected.

## Physical setup

1. Connect the instrument's Ethernet port to a network interface of the computer that serves nothing else, for example a USB Ethernet adapter. Never connect the instrument to a shared network: it runs its own DHCP server, which gives the computer's interface its address. The instrument's address on that link is the `host` below; it is the DHCP server of that interface (`ipconfig /all` on Windows lists it).
2. On the bottle station, fill the Sheath, PLUS One and PLUS Two bottles and empty the Waste bottle. In the deck cartridges, fill the Buffer, Decon, Clean and Water troughs.
3. Switch the instrument on and wait until its internal computer has started. The detector may stay off: `start_up()` below switches it on.
4. Close any other client software, for example the instrument's own software.

## Setup

Create the instrument with the `IQue` factory and the instrument's address, and call `setup()`. Setup connects to the controller, checks its version and that the instrument is an iQue Screener PLUS, downloads the instrument's calibration file, and resets and initialises the devices. The probe goes first: it rises and parks, and setup goes on only once it is at the top and its contact sensor reports no contact. Then the shaker homes and the plate rail moves out. When the controller still holds a finished worklist that was never collected, setup logs a warning (see Interrupted reads below).

`calibration_backup_dir` is a folder on this computer: before PyLabRobot stores a new calibration on the instrument, it copies the instrument's file there (see Calibration below).

This cell needs the instrument. To follow the guide without one, run the simulation cell below instead.

In [ ]:
import pathlib

from pylabrobot.sartorius import IQue

ique = IQue(
  name="ique",
  host="192.0.2.10",  # replace with your instrument's address
  calibration_backup_dir=pathlib.Path("ique-calibration-backups"),
)
await ique.setup()

### Simulation

`IQue(simulation=True)` builds the instrument around a simulated controller that runs in the Python process, so every cell of this guide runs without an instrument. The simulator has the controller's devices and refusals, runs worklists on a clock `simulation_speed` times faster than real time, and sends a burst of events for every sampled well. Its calibration places every position at the nominal place an instrument's teaching starts from. Run this cell instead of the one above.

In [ ]:
import pathlib

from pylabrobot.sartorius import IQue

ique = IQue(
  name="ique",
  simulation=True,
  simulation_speed=1000,
  calibration_backup_dir=pathlib.Path("ique-calibration-backups"),
)
await ique.setup()

## Status

Setup reads which instrument it is: its system type (`iQue3` and `iQue3HD` have the VBR configuration, `iQuePlusVYB` and `iQuePlusBR` the VYB and BR ones) and its system ID. `driver.request_status()` asks for the system state (`Green`; `Yellow` while a device reports a warning; `Red` while one is faulted or disconnected), the state of every device, the probe's contact sensor and whether a worklist runs.

In [ ]:
status = await ique.driver.request_status()
print(ique.driver.system_type, ique.driver.system_id, status.system_state)
{kind: info.state for kind, info in status.devices.items()}

The detector reports whether a measurement can start, and it reports its fluid levels and its laser and pressure states as status messages.

In [ ]:
detector_status = await ique.detector.request_status()
print(detector_status.ready_to_run)
[message.message for message in detector_status.messages]

## Plate rail and plate holder

The plate holder sits on the plate rail, which carries it into the instrument. A plate goes on or comes off only while the rail is out: assigning a plate to the holder while the rail is in raises `PlateAccessError`. The rail is out after setup and after every read, and a read moves it in by itself. To move the rail out, the probe must be parked.

In [ ]:
await ique.plate_rail.eject()

Put the plate on the plate holder, then assign it to the holder. A read takes the plate's format, 96, 384 or 1536 wells, from its grid of wells.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

plate = cor_96_wellplate_360uL_Fb(name="plate")
ique.plate_holder.assign_child_resource(plate)

## Reading a plate

`read_plate` samples every well, row by row. With the default settings the probe stays 1 s in each well and 0.5 s above it, the pump runs at 29 rpm, and the plate is shaken for 4 s after every 12 wells while the probe dips in Buffer. The instrument moves the plate in, checks the deck height, samples the wells, flushes the line while it moves the plate out, cleans the probe in Decon, Clean and Water and stops. A 96-well plate takes about 7.5 minutes.

Before anything is sent, the read is checked: the plate is on the plate holder; the calibration holds the plate's format, the stations and the deck height, and the plate's model mapped for the format; no well is sampled below the deck height; no worklist runs; the detector is ready to run; the plate rail is ready; the probe reports no contact. Name the plate's model in the calibration with `plate_model`: the probe samples each well at the height the model's depth map gives, and a plate whose well bottoms sit higher than the model's would be hit. A plate resource whose `model` is the name of a calibrated model needs no `plate_model`. The simulated calibration's model for 96-well plates is `Generic 96 Well Plate`; the Calibration section below lists the models of an instrument. `metadata`, a list of (key, value) pairs, goes into the FCS file.

In [ ]:
plate_model = "Generic 96 Well Plate"  # a plate model of your instrument's calibration
read = await ique.read_plate(plate, plate_model=plate_model)

### Wells

The events are assigned to wells by when the wells were sampled: each well shows up as a burst of events, followed by the quiet of the air gap. `verdict` is `identified` when every well's burst was found and fits the timing of its sampling; it is `needs_review` when a well had to be placed by timing alone, its burst does not fit its timing, or the wells of a run could be shifted by a well or two. An empty well at either end of a run, or sample left in the line before the first well, looks the same as such a shift, so those wells are named for review. `message` names those wells, and the read logs it as a warning.

In [ ]:
read.verdict, read.message

The wells are placed from the first burst of events. A measured transit delay, the time from the start of a well's sampling command to its first events at the detector, places the first well where that time predicts instead, so that an empty first well or sample left in the line cannot shift the wells. `transit_delays()` gives it for every well that was found; their median, from a read whose verdict is `identified`, is the value for `transit_delay` in the read settings below, at the same pump speed.

In [ ]:
import statistics

transit = round(statistics.median(read.transit_delays().values()), 1)
transit

`event_counts()` gives the number of events of every well, in sampling order.

In [ ]:
counts = read.event_counts()
counts["A01"], counts["H12"]

`well_events` returns the events of one well, with the times they have in the plate's acquisition. Each event holds every parameter (`FSC-H`, `SSC-H`, `BL1-H`, ..., `Width`, `Time`); `parameter` returns one of them for every event, and `to_numpy()` the events as an array when numpy is installed.

In [ ]:
events = read.well_events("A1")
events.event_count, events.parameter_names[:4], events.parameter("FSC-H")[:3]

`write_fcs` writes the controller's FCS file of the plate as it is, and one FCS file per well (`Well_A01.fcs`, ...) with the well's events, the instrument's keywords and the well's name. Files that exist are not replaced unless you pass `overwrite=True`.

In [ ]:
paths = read.write_fcs("plate-fcs")
paths[:3]

## Read settings

`PlateReadSettings` holds how a plate is read; every value is checked against the instrument's ranges when it is created. Times are in seconds, speeds in rpm.

- **Sampling**: `sip_time`, how long the probe stays in each well (0.5-59.9 s); `up_time`, how long it then waits above the well, drawing the air gap between two wells (0.5-59.9 s; the lower limit stands until the timing of the probe's rise against the plate's movement has been measured on an instrument); `pump_speed` (10-32 rpm); `sampling_height` above the mapped well bottom.
- **Shaking**: `shake_mode="after_wells"` shakes the plate for `shake_duration` at `shake_speed` after every `shake_every` wells while the probe dips in `shake_station`; `"continuous"` shakes a 96-well plate throughout the read; `"none"` does not shake. `pre_plate_shake` shakes once before the first well.
- **Rinses**: with `probe_rinse`, the probe is rinsed after every `rinse_every` wells in the `rinses` steps, each a station and a duration.
- **Flush and cleans**: after the last well the line is flushed in Buffer for `flush_duration`, then, with `post_plate_clean`, the probe is cleaned in the `cleans` steps. Flush and cleans together must last long enough to clear the line at the pump speed. `pre_plate_prime` primes the line before the first well.
- **Well identification**: `transit_delay`, the measured transit delay at this pump speed (see Wells above), or None to place the wells from the first burst of events.
- **Acquisition**: `thresholds`, one or two `TriggerThreshold`s, the first on `FSC-H` or `SSC-H`; `lasers`, a combination the instrument's configuration offers (blue and red on a VBR or BR instrument, as here), or None for all of them; `event_limit` and `event_maximum`.

In [ ]:
from pylabrobot.sartorius.ique import PlateReadSettings, RinseStep, TriggerThreshold

settings = PlateReadSettings(
  sip_time=1.5,
  up_time=0.6,
  shake_mode="after_wells",
  shake_every=8,
  shake_duration=5,
  shake_speed=2000,
  probe_rinse=True,
  rinse_every=24,
  rinses=(RinseStep("Water", 1.0),),
  cleans=(RinseStep("Decon", 30), RinseStep("Clean", 30), RinseStep("Water", 60)),
  thresholds=(TriggerThreshold("FSC-H", 100_000), TriggerThreshold("BL1-H", 2_000)),
  lasers=("blue", "red"),
  transit_delay=transit,
)

Pass the settings to the read.

In [ ]:
read = await ique.read_plate(plate, plate_model=plate_model, settings=settings)
read.verdict

## Wells and their order

`wells` reads some of the wells, given as wells of the plate or as names (`"A1"` or `"A01"`). `order` sets the order they are sampled in: `"row"`, `"serpentine_row"` (row A left to right, row B right to left, ...), `"column"` or `"serpentine_column"`; `order=None` samples them in the order given. `read.wells` lists the sampled wells in sampling order.

In [ ]:
read = await ique.read_plate(
  plate, plate_model=plate_model, wells=plate.get_items("A1:B12"), order="serpentine_row"
)
read.wells[:14]

## Stopping a read early

A read calls `on_progress` for every command of its worklist that starts or ends, with the command, its position and the events received so far. Here it notes when well A12 has been sampled.

In [ ]:
import asyncio

from pylabrobot.sartorius.ique.driver.contract import SampleWellCommand

row_a_sampled = asyncio.Event()


def on_progress(progress):
  """Note when well A12 has been sampled."""
  command = progress.command
  if progress.phase == "completed" and isinstance(command, SampleWellCommand):
    if (command.row, command.column) == ("A", 12):
      row_a_sampled.set()

`stop_read_early` stops the read that runs and keeps what it has read: the well being sampled is finished, the other wells are skipped, and the line is flushed and the probe cleaned as at the end of every read. The read then returns the wells sampled until then, with `stopped_early` set. When the stop comes during the last well, or once the flush has started, nothing is skipped: the read ends as usual. A read that has not started recording yet is cancelled instead, and raises `WorklistCancelledError`.

In [ ]:
task = asyncio.ensure_future(
  ique.read_plate(plate, plate_model=plate_model, on_progress=on_progress)
)
await row_a_sampled.wait()
await ique.stop_read_early()
read = await task
read.stopped_early, read.wells

## Reading a tube

Put a 1.5 mL microcentrifuge tube with the sample in the tube holder. `read_tube` feels for the tube (the read fails when the holder is empty), primes the line, draws from the tube for `sip_time`, flushes the line and cleans the probe; every event belongs to the tube. `TubeReadSettings` takes the acquisition settings of a plate read, the pump speed and the cleans. With a 30 s sip a read takes about 4.5 minutes. `stop_read_early` stops a tube read that records as it stops a plate read: the draw from the tube finishes, then the line is flushed and the probe cleaned.

In [ ]:
from pylabrobot.sartorius.ique import TubeReadSettings

tube = await ique.read_tube(TubeReadSettings(sip_time=30.0))
tube.event_count

## Interrupted reads

However a read ends, whether it completes, stops early, fails, times out or its task is cancelled, `ique.flush_needed` records whether it left sample in the line or skipped its probe cleans (see Prime below). Cancelling the task of a read cancels the worklist on the instrument, and `ique.driver.interrupted_run` then holds what the read had recorded.

When the session ends while a worklist runs, for example because the connection fails, the controller finishes the worklist and holds it, with its FCS file, until it is finalised; until then it runs no other worklist, and setup logs a warning. `recover_unfinalized_run()` downloads the run's FCS file, rebuilds the plate read when it was one (pass the `settings` it was read with), and finalises the worklist; it returns None when the controller holds none. `finalize_worklist()` finalises it without downloading anything, and every read and maintenance method takes `finalize_previous=True` to do that first.

In [ ]:
recovered = await ique.recover_unfinalized_run()
recovered is None

## Maintenance

### Start-up

Start the day with `start_up()`: it switches the detector on if it is off, waits until the detector can start maintenance, and cleans the probe and the tubing for 20 minutes (`clean_duration`). The Waste bottle must be empty, the other bottles and the Decon, Clean and Water troughs filled. Like every maintenance routine below, it returns the completed run, with the start and end time of every command.

In [ ]:
run = await ique.start_up()

### Probe and tubing clean

The probe dips in Decon for a quarter of the duration, in Clean for a quarter and in Water for half. The duration is 4 to 32 minutes, in steps of 4 minutes, given in seconds.

In [ ]:
run = await ique.clean_probe_and_tubing(duration=480)

### Prime

`prime()` fills the line from Buffer, the probe dipping repeatedly, for 60 s by default. After a read that ended with sample in the line, however it ended, `ique.flush_needed` is True until a prime, a probe and tubing clean, start-up or a read that flushed and cleaned. It is also True after a read whose probe cleans were skipped; run a probe and tubing clean then.

In [ ]:
run = await ique.prime()

### Detector cleans

The detector runs its own maintenance routines, with the sampler tubing installed: `quick_clean()` (about 11 minutes), `long_clean()` (about 30 minutes), `unclog()` (about 10.5 minutes, when the sample line clogs and once a month) and `debubble()` (about 13 minutes, the detector primes from its bottles). Each returns the detector to standby at its end. They need the detector ready for fluidics maintenance rather than ready to run, so they also run while a fault blocks measurements.

In [ ]:
run = await ique.detector.quick_clean()

### Fluidics check

After changing filters, the FluidLink or the probe, run the one-minute fluidics check: the probe stays in Water while the pump runs. Meanwhile, look for leaks at the pump clamp, the tubing port and both ends of the FluidLink.

In [ ]:
run = await ique.fluidics_check()

### Deck-height check

Every plate read starts by touching a test point on the plate holder with the probe, and fails unless the contact is within the tolerance the controller states, 0.25 mm, of the calibrated deck height. `validate_deck_height()` runs that check on its own.

In [ ]:
run = await ique.validate_deck_height()

### Shut-down

End the day with `shut_down()`: it cleans the probe and the tubing, then shuts the detector down. The bottles and troughs must be prepared as for start-up. Nothing is run when the detector is off already; `ique.detector.power_off()` shuts the detector down without the clean.

In [ ]:
run = await ique.shut_down()

## Calibration

The instrument keeps one calibration file, which holds every place the probe goes to: the deck height and the spot where it is measured, the corners of each plate format, the stations and the tube holder, and the plate models with the heights of their well bottoms. Every depth in it is stored relative to the deck height. PyLabRobot downloads the file at setup and builds every worklist from it; it never clears it. `driver.calibration` holds what it read.

In [ ]:
calibration = ique.driver.calibration
print(calibration.instrument.deck_height, calibration.instrument.deck_height_calibrated_on)
[model.name for model in calibration.plate_models]

### Deck-height calibration

After a probe change, measure the deck height again. `calibrate_deck_height(confirm=True)` parks the probe, moves the plate rail in, lowers the probe at the test point until it touches, and accepts a contact within the tolerance the controller states for a new probe, 4 mm, of the stored deck height. The probe then parks and the rail moves out. PyLabRobot copies the instrument's calibration file to `calibration_backup_dir`, changes only the deck height and the date of its measurement in that file, leaving every other byte as it was, stores it on the instrument, and reads it back to check it. Since every depth follows the deck height, the change applies to every well, station and tube.

The method refuses without `confirm=True`, because it changes the stored calibration, and without a backup folder. Teaching plate corners, stations and plate models is not offered.

In [ ]:
await ique.calibrate_deck_height(confirm=True)

## Teardown

The plate rail is out after every read: take the plate off the holder.

In [ ]:
ique.plate_holder.unassign_child_resource(plate)

`stop()` stops the pump and the shaker, raises the probe until its contact sensor reports no contact, parks it and closes the session with the controller. When the sensor still reports contact, it raises `UnsafeMotionError` and leaves the session open: free the probe and call it again. It refuses while a worklist runs, unless you pass `cancel=True`.

In [ ]:
await ique.stop()